# Fair per-check rescore (RA-ML-FAIRNESS-PER-CHECK-SCORERS, RA-ML-PAYMENT-DUP-RESCORE)
Rescores every `gold_anomaly_scores` row whose check type is **not** handled by the core `score.py` path, using the **champion signature columns** (no `arpu_tier_weight`).
* Keeps the existing `exception_id`, so downstream joins don't change.
* Builds the same 27 features as `feature_build.py`, with exception attributes from `gold_anomaly_scores` and severity from `v_gold_leakage_summary`.
* Applies composite / rank **per check_type**; `review_priority` is stratified by **check_type x segment_classification x arpu_tier** (top 5% = HIGH, next 15% = MEDIUM; DEC-PM-CREDIT-PROXY-B4).
* Deletes rows with no match in `v_gold_leakage_summary` (the 710 stale `payment_duplicate` rows).
* Writes only to `cdm_tmforum.revenue_assurance`. Run it on serverless with environment client 4 or later (the v64 model needs numpy 2).

In [0]:
import mlflow
from mlflow import MlflowClient
from pyspark.sql import functions as F, Window

CAT, SCH = "cdm_tmforum", "revenue_assurance"
M = f"{CAT}.{SCH}.ra_anomaly_isolation_forest"
STAGE = f"{CAT}.{SCH}._staging_fair_rescore_per_check"
mlflow.set_registry_uri("databricks-uc")
FEATS = mlflow.models.get_model_info(f"models:/{M}@champion").signature.inputs.input_names()
VER = MlflowClient().get_model_version_by_alias(M, "champion").version
assert "arpu_tier_weight" not in FEATS, "champion still uses arpu_tier_weight"
print(f"champion v{VER}, {len(FEATS)} features")

core = [r[0] for r in spark.sql(f"SELECT DISTINCT check_type FROM {CAT}.{SCH}.feature_account_anomaly").collect()]
core_sql = ",".join(f"'{c}'" for c in core) if core else "''"

feat_df = spark.sql(f"""
WITH g AS (
  SELECT exception_id, item_id, check_type, customer_id, account_name, reference_id, source_table,
         detection_method, known_leakage_flag, CAST(amount_at_risk AS DOUBLE) AS amount_at_risk
  FROM {CAT}.{SCH}.gold_anomaly_scores WHERE check_type NOT IN ({core_sql})),
v AS (SELECT check_type, reference_id, MAX(severity) AS severity
      FROM {CAT}.{SCH}.v_gold_leakage_summary GROUP BY 1, 2),
e AS (SELECT g.*, v.severity FROM g JOIN v ON g.check_type = v.check_type AND g.reference_id = v.reference_id),
ctr AS (SELECT check_type, CAST(COUNT(*) AS DOUBLE) ct_feedback_count,
   SUM(CASE WHEN outcome='RECOVERED' THEN 1 ELSE 0 END)/GREATEST(COUNT(*),1) ct_recovery_rate,
   SUM(CASE WHEN outcome='WRITTEN_OFF' THEN 1 ELSE 0 END)/GREATEST(COUNT(*),1) ct_writeoff_rate,
   SUM(CASE WHEN outcome='FALSE_POSITIVE' THEN 1 ELSE 0 END)/GREATEST(COUNT(*),1) ct_false_positive_rate,
   AVG(recovery_pct) ct_avg_recovery_pct, AVG(investigation_hours) ct_avg_investigation_hours,
   AVG(CAST(triage_was_accurate AS INT)) ct_triage_accuracy
   FROM {CAT}.{SCH}.recovery_feedback GROUP BY check_type),
cr AS (SELECT customer_id, CAST(COUNT(*) AS DOUBLE) cust_feedback_count,
   SUM(CASE WHEN outcome='RECOVERED' THEN 1 ELSE 0 END)/GREATEST(COUNT(*),1) cust_recovery_rate,
   SUM(CASE WHEN outcome='WRITTEN_OFF' THEN 1 ELSE 0 END)/GREATEST(COUNT(*),1) cust_writeoff_rate,
   AVG(recovery_pct) cust_avg_recovery_pct, AVG(investigation_hours) cust_avg_investigation_hours
   FROM {CAT}.{SCH}.recovery_feedback WHERE customer_id IS NOT NULL GROUP BY customer_id)
SELECT e.*,
 CAST(LOG1P(GREATEST(e.amount_at_risk, 0.0)) AS DOUBLE) amount_at_risk_log,
 CAST(CASE WHEN UPPER(e.severity) IN ('HIGH','CRITICAL') THEN 2.0 ELSE 1.0 END AS DOUBLE) severity_weight,
 CAST(CASE WHEN e.customer_id IS NULL THEN 1.0 ELSE 0.0 END AS DOUBLE) missing_customer_flag,
 CAST(CASE WHEN e.detection_method = 'ai_extracted' THEN 1.0 ELSE 0.0 END AS DOUBLE) ai_extracted_flag,
 CAST(COUNT(*) OVER (PARTITION BY e.check_type) AS DOUBLE) check_type_frequency,
 CAST(COUNT(*) OVER (PARTITION BY e.customer_id) AS DOUBLE) customer_exception_count,
 CAST(SUM(e.amount_at_risk) OVER (PARTITION BY e.customer_id) AS DOUBLE) customer_total_amount_at_risk,
 CAST(AVG(e.amount_at_risk) OVER (PARTITION BY e.customer_id) AS DOUBLE) customer_avg_amount_at_risk,
 CAST(COALESCE(s.composite_health_score, 100.0) AS DOUBLE) composite_health_score,
 CAST(COALESCE(s.total_exceptions, 0.0) AS DOUBLE) scorecard_total_exceptions,
 CAST(COALESCE(s.total_amount_at_risk, 0.0) AS DOUBLE) scorecard_total_amount_at_risk,
 CAST(CASE COALESCE(s.risk_tier,'GREEN') WHEN 'RED' THEN 2.0 WHEN 'AMBER' THEN 1.0 ELSE 0.0 END AS DOUBLE) risk_tier_weight,
 CAST(CASE WHEN COALESCE(c.credit_class,'') IN ('d','unrated') THEN 2.0
           WHEN COALESCE(c.credit_class,'') IN ('c','cc','ccc') THEN 1.5
           WHEN COALESCE(c.credit_class,'') IN ('b','bb','bbb') THEN 1.0 ELSE 0.0 END AS DOUBLE) credit_risk_weight,
 CAST(COALESCE(ctr.ct_feedback_count, 0.0) AS DOUBLE) ct_feedback_count,
 CAST(COALESCE(ctr.ct_recovery_rate, 0.5) AS DOUBLE) ct_recovery_rate,
 CAST(COALESCE(ctr.ct_writeoff_rate, 0.0) AS DOUBLE) ct_writeoff_rate,
 CAST(COALESCE(ctr.ct_false_positive_rate, 0.0) AS DOUBLE) ct_false_positive_rate,
 CAST(COALESCE(ctr.ct_avg_recovery_pct, 0.5) AS DOUBLE) ct_avg_recovery_pct,
 CAST(COALESCE(ctr.ct_avg_investigation_hours, 8.0) AS DOUBLE) ct_avg_investigation_hours,
 CAST(COALESCE(ctr.ct_triage_accuracy, 0.5) AS DOUBLE) ct_triage_accuracy,
 CAST(COALESCE(cr.cust_feedback_count, 0.0) AS DOUBLE) cust_feedback_count,
 CAST(COALESCE(cr.cust_recovery_rate, 0.5) AS DOUBLE) cust_recovery_rate,
 CAST(COALESCE(cr.cust_writeoff_rate, 0.0) AS DOUBLE) cust_writeoff_rate,
 CAST(COALESCE(cr.cust_avg_recovery_pct, 0.5) AS DOUBLE) cust_avg_recovery_pct,
 CAST(COALESCE(cr.cust_avg_investigation_hours, 8.0) AS DOUBLE) cust_avg_investigation_hours,
 CAST(0.4*COALESCE(1.0-ctr.ct_recovery_rate,0.5) + 0.3*COALESCE(ctr.ct_writeoff_rate,0.0)*3.0
      + 0.2*COALESCE(cr.cust_writeoff_rate,0.0)*3.0 + 0.1*LEAST(COALESCE(ctr.ct_avg_investigation_hours,8.0)/15.0,1.0) AS DOUBLE) recovery_difficulty_score
FROM e
LEFT JOIN {CAT}.{SCH}.gold_reconciliation_scorecard s ON e.customer_id = s.customer_id
LEFT JOIN {CAT}.tmf_customer.customer c ON e.customer_id = c.customer_id
LEFT JOIN ctr ON e.check_type = ctr.check_type
LEFT JOIN cr ON e.customer_id = cr.customer_id
""").fillna(0.0, subset=FEATS)

udf = mlflow.pyfunc.spark_udf(spark, f"models:/{M}@champion", result_type="double", env_manager="local")
scored = feat_df.withColumn("isolation_forest_score", udf(*[F.col(c).cast("double") for c in FEATS]))

w_ct = Window.partitionBy("check_type")
scored = (scored
  .withColumn("_sm", F.avg("isolation_forest_score").over(w_ct))
  .withColumn("_ss", F.stddev_pop("isolation_forest_score").over(w_ct))
  .withColumn("_am", F.avg("amount_at_risk_log").over(w_ct))
  .withColumn("_as", F.stddev_pop("amount_at_risk_log").over(w_ct))
  .withColumn("zscore", F.when(F.col("_ss") > 0, (F.col("isolation_forest_score") - F.col("_sm")) / F.col("_ss")).otherwise(F.lit(0.0)))
  .withColumn("_az", F.when(F.col("_as") > 0, (F.col("amount_at_risk_log") - F.col("_am")) / F.col("_as")).otherwise(F.lit(0.0)))
  .withColumn("composite_anomaly_score", 0.75 * F.greatest("zscore", F.lit(0.0)) + 0.25 * F.greatest("_az", F.lit(0.0)))
  .withColumn("anomaly_rank", F.rank().over(Window.partitionBy("check_type").orderBy(F.col("composite_anomaly_score").desc())))
  # RA-ML-WITHIN-CHECK-FAIRNESS / DEC-PM-CREDIT-PROXY-B4: priority stratified by check_type x segment x arpu_tier
  .join(spark.table(f"{CAT}.tmf_customer.customer").select("customer_id", "segment_classification", "arpu_tier"), "customer_id", "left")
  .withColumn("_frac", F.rank().over(Window.partitionBy("check_type", "segment_classification", "arpu_tier")
                                     .orderBy(F.col("composite_anomaly_score").desc(), F.col("isolation_forest_score").desc()))
                       / F.count(F.lit(1)).over(Window.partitionBy("check_type", "segment_classification", "arpu_tier")))
  .withColumn("review_priority", F.when(F.col("_frac") <= 0.05, "HIGH").when(F.col("_frac") <= 0.20, "MEDIUM").otherwise("LOW"))
  .select("exception_id", "item_id", "check_type", "customer_id", "account_name", "reference_id", "source_table",
          "detection_method", "known_leakage_flag", "amount_at_risk", "isolation_forest_score",
          F.round("zscore", 6).alias("zscore"), "composite_anomaly_score", F.col("anomaly_rank").cast("int").alias("anomaly_rank"),
          "review_priority", F.lit(M).alias("registered_model_name"), F.lit("champion").alias("registered_model_alias"),
          F.current_timestamp().alias("scored_at")))

scored.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(STAGE)
spark.sql(f"COMMENT ON TABLE {STAGE} IS 'Staging: fair per-check rescore of non-core gold_anomaly_scores rows with the champion signature (no arpu_tier_weight). Written by mlops/rescore_per_check_fair.'")
n = spark.table(STAGE).count()
print(f"staged {n:,} rows")

In [0]:
def gaps(table, where="1=1"):
    out = {}
    for dim in ["arpu_tier", "segment_classification"]:
        rows = spark.sql(f"""SELECT c.{dim} grp, AVG(CASE WHEN g.review_priority='HIGH' THEN 1.0 ELSE 0.0 END) hr, COUNT(*) n
            FROM {table} g JOIN {CAT}.tmf_customer.customer c ON g.customer_id = c.customer_id
            WHERE {where} GROUP BY 1""").collect()
        rates = {r.grp: float(r.hr) for r in rows}
        out[dim] = (max(rates.values()) - min(rates.values()), rates)
    return out

before = gaps(f"{CAT}.{SCH}.gold_anomaly_scores")
after_stage = gaps(STAGE)
for k in before:
    print(f"{k}: table-wide gap before {before[k][0]:.4f} | staged non-core gap {after_stage[k][0]:.4f}")
    print("   staged rates:", {g: round(v, 4) for g, v in after_stage[k][1].items()})

# Gate = non-regression: only merge if staged gaps are below the current table-wide gaps.
# Absolute targets (0.03 / 0.04) are reported. Residual gap is driven by credit_risk_weight,
# which is a 1:1 proxy for arpu_tier in this data (see RA-ML-CREDIT-RISK-PROXY-RETRAIN).
TARGET_ARPU, TARGET_SEG = 0.03, 0.04
for k, tgt in [("arpu_tier", TARGET_ARPU), ("segment_classification", TARGET_SEG)]:
    print(f"{k}: staged {after_stage[k][0]:.4f} vs target {tgt} -> {'MET' if after_stage[k][0] <= tgt else 'NOT MET'}")
assert all(after_stage[k][0] < before[k][0] for k in before), "Fairness non-regression gate FAILED: not merging"

In [0]:
pre = spark.table(f"{CAT}.{SCH}.gold_anomaly_scores").count()
spark.sql(f"""
MERGE INTO {CAT}.{SCH}.gold_anomaly_scores t
USING {STAGE} s ON t.exception_id = s.exception_id
WHEN MATCHED THEN UPDATE SET
  t.isolation_forest_score = s.isolation_forest_score, t.zscore = s.zscore,
  t.composite_anomaly_score = s.composite_anomaly_score, t.anomaly_rank = s.anomaly_rank,
  t.review_priority = s.review_priority, t.registered_model_name = s.registered_model_name,
  t.registered_model_alias = s.registered_model_alias, t.scored_at = s.scored_at
""")
# RA-ML-PAYMENT-DUP-RESCORE: remove rows that no longer exist in v_gold_leakage_summary
# A correlated DELETE against the masked v_gold view fails with UNSUPPORTED_NESTED_ROW_OR_COLUMN_ACCESS_POLICY,
# so stage the stale ids first, then delete by id.
stale = spark.sql(f"""SELECT g.exception_id FROM {CAT}.{SCH}.gold_anomaly_scores g LEFT ANTI JOIN
  (SELECT DISTINCT check_type, reference_id FROM {CAT}.{SCH}.v_gold_leakage_summary) v
  ON g.check_type = v.check_type AND g.reference_id = v.reference_id""")
stale.write.mode("overwrite").saveAsTable(f"{CAT}.{SCH}._staging_stale_anomaly_ids")
spark.sql(f"DELETE FROM {CAT}.{SCH}.gold_anomaly_scores WHERE exception_id IN (SELECT exception_id FROM {CAT}.{SCH}._staging_stale_anomaly_ids)")
post = spark.table(f"{CAT}.{SCH}.gold_anomaly_scores").count()
orphans = spark.sql(f"""SELECT COUNT(*) FROM {CAT}.{SCH}.gold_anomaly_scores g LEFT ANTI JOIN
  (SELECT DISTINCT check_type, reference_id FROM {CAT}.{SCH}.v_gold_leakage_summary) v
  ON g.check_type = v.check_type AND g.reference_id = v.reference_id""").first()[0]
final = gaps(f"{CAT}.{SCH}.gold_anomaly_scores")
print(f"rows {pre:,} -> {post:,}; orphans vs v_gold = {orphans}")
for k in final:
    print(f"{k}: table-wide gap {before[k][0]:.4f} -> {final[k][0]:.4f}")
dbutils.notebook.exit(f"champion=v{VER} staged={n} pre={pre} post={post} orphans={orphans} "
                      f"arpu_gap={before['arpu_tier'][0]:.4f}->{final['arpu_tier'][0]:.4f} "
                      f"seg_gap={before['segment_classification'][0]:.4f}->{final['segment_classification'][0]:.4f}")